## Limpieza de timingsys.com (Cronomancha)

`scrape_timingsys.ipynb` deja una fila **por carrera** (no por participante) con el recuento real de finishers por sexo (`finisher_h`, `finisher_d`) ya sumado desde la API de Cronomancha. Igual que en `cronorunner`, la fuente entrega casi todo hecho y el trabajo que queda es:

1. `nombre_carrera`: ya viene un nombre por carrera (cada modalidad/recorrido de un mismo evento es una fila distinta, p.ej. `... - CORTA`, `... - EBIKE`), así que solo hay que normalizar espacios.
2. `distancia` en km (la fuente trae metros; en 11 carreras no hay distancia y se deja en `0`, el mismo centinela de "desconocida" que usan las otras 12 fuentes y que `Limpieza_outliers.ipynb` convierte después en `NaN`).
3. `fecha` (llega en dos formatos mezclados, con y sin hora: `2026-09-05T00:00:00` y `2025-12-31`) y `dia_semana`.
4. `tipo_modalidad`: esta fuente sí trae una `modalidad` propia (`A PIE`, `BICI (MTB/BTT)`, `TRAIL`, `DUATLON/TRIATLON`), así que la usamos como pista principal y solo recurrimos a palabras clave del nombre cuando la modalidad no basta (p.ej. distinguir `marcha` o `trail running` dentro de `A PIE`). Cronomancha cronometra mucha prueba ciclista en Castilla-La Mancha (casi la mitad de las filas), que entra en la categoría `Ciclismo y btt` como en el resto de fuentes.
5. `publico`, por palabras clave del nombre (mismo criterio que en `cronorunner`).
6. `municipio`/`comarca`/`provincia`: `lat`/`lon` solo existen para 107 de los 379 eventos, y el catálogo (`timingsys_catalogo_eventos.csv`) trae un `lugar` de texto libre para esos mismos 107 y ninguno más. Para el resto (~72 % de los eventos) hay que sacar la población del propio nombre del evento, con validación para no aceptar geocodificaciones inventadas.

Todas las funciones de más abajo se prueban al final del notebook contra filas reales del CSV antes de fiarnos de ellas con la fuente completa.

In [1]:
import re
import pandas as pd

RUTA_SUCIO = "../../data/raw/timingsys/DF_TIMINGSYS_SUCIO.csv"
RUTA_CATALOGO = "../../data/raw/timingsys/timingsys_catalogo_eventos.csv"

df = pd.read_csv(RUTA_SUCIO, encoding="utf-8-sig")

# El catálogo trae un "lugar" de texto libre por evento (solo cuando el organizador lo rellenó).
catalogo = pd.read_csv(RUTA_CATALOGO, usecols=["id_timingsys", "lugar"]).drop_duplicates("id_timingsys")
df = df.merge(catalogo, on="id_timingsys", how="left", validate="many_to_one")

print(df.shape)
print("Eventos distintos:", df["id_timingsys"].nunique())
df.head()

(394, 14)
Eventos distintos: 379
   id_timingsys  ...                  lugar
0           767  ...                    NaN
1           766  ...                    NaN
2           765  ...                    NaN
3           797  ...    LA ALMARCHA, CUENCA
4           757  ...  FUENTELESPINO DE HARO

[5 rows x 14 columns]


### Funciones de limpieza

El nombre de la carrera se queda tal cual salvo por los espacios (hay nombres con espacio final o doble). `distancia_metros` es exacto en la fuente, así que solo se pasa a km; los nulos pasan a `0` para respetar el centinela común.

In [3]:
def limpiar_nombre_carrera(serie: pd.Series) -> pd.Series:
    return serie.str.replace(r"\s+", " ", regex=True).str.strip()


def derivar_distancia_km(df_in: pd.DataFrame) -> pd.Series:
    # NaN -> 0: centinela de "distancia desconocida" común a todas las fuentes
    return (pd.to_numeric(df_in["distancia_metros"], errors="coerce") / 1000.0).fillna(0.0)


_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]


def derivar_fecha_y_dia_semana(df_in: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    # La fuente mezcla "2026-09-05T00:00:00" (38 filas) con "2025-12-31" (356): sin
    # format="ISO8601" pandas infiere el formato de la primera fila y deja el resto en NaT.
    fecha = pd.to_datetime(df_in["fecha_evento"], format="ISO8601", errors="coerce")
    dia_semana = fecha.dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))
    return fecha, dia_semana


def clasificar_tipo_modalidad(nombre, modalidad) -> str:
    """La `modalidad` de Cronomancha manda cuando es concluyente (bici, duatlón/triatlón);
    si no, decide el nombre por palabras clave, igual que en las otras fuentes."""
    mod = "" if pd.isna(modalidad) else modalidad.upper()
    texto = "" if pd.isna(nombre) else nombre.lower()
    if "BICI" in mod:
        return "Ciclismo y btt"
    if "DUATLON" in mod or "TRIATLON" in mod:
        return "Multidisciplina"
    if re.search(r"esqu[ií]", texto):
        return "Otros"
    if re.search(r"duatl|triatl|triathlon|acuatl|aquathlon", texto):
        return "Multidisciplina"
    if re.search(r"\bbtt\b|\bmtb\b|\bbike\b|ciclis|ciclo", texto):
        return "Ciclismo y btt"
    if "TRAIL" in mod or re.search(r"trail|cross|vertical|\bkv\b", texto):
        return "trail running"
    if re.search(r"marcha", texto):
        return "marcha"
    return "road running"


def clasificar_publico(nombre) -> str:
    texto = "" if pd.isna(nombre) else nombre.lower()
    if re.search(r"equipos?\b", texto):
        return "Equipos"
    if re.search(r"veteran|m[aá]ster|\bsenior\b", texto):
        return "Mayores/Veteranos"
    if re.search(r"\belite\b|[ée]lite|profesional", texto):
        return "Elite"
    if re.search(
        r"infantil|alev[ií]n|benjam|prebenjam|cadet|juvenil|j[uú]nior|menores|promesa|escolar|chupet|peques?\b",
        texto,
    ):
        return "Infantil"
    return "Absoluta/General"

### Aplicar las funciones a la fuente completa

In [5]:
df["nombre_carrera"] = limpiar_nombre_carrera(df["nombre_carrera"])
df["nombre_evento"] = limpiar_nombre_carrera(df["nombre_evento"])
df["distancia"] = derivar_distancia_km(df)
df["fecha"], df["dia_semana"] = derivar_fecha_y_dia_semana(df)
df["tipo_modalidad"] = [clasificar_tipo_modalidad(n, m) for n, m in zip(df["nombre_carrera"], df["modalidad"])]
df["publico"] = df["nombre_carrera"].apply(clasificar_publico)

# finisher_h/finisher_d ya vienen como enteros reales de la API; comprobamos que cuadran con el total.
assert (df["finisher_h"] + df["finisher_d"] == df["total_finishers"]).all()
assert df["fecha"].notna().all()

print(df["tipo_modalidad"].value_counts())
print()
print(df["publico"].value_counts())
print()
print("Filas con distancia desconocida (0):", (df["distancia"] == 0).sum())

tipo_modalidad
Ciclismo y btt     175
road running       165
trail running       45
Multidisciplina      5
marcha               4
Name: count, dtype: int64

publico
Absoluta/General    389
Infantil              4
Elite                 1
Name: count, dtype: int64

Filas con distancia desconocida (0): 11


### Ubicación -- coordenadas, `lugar` del catálogo o nombre del evento

Tres fuentes de ubicación, por orden de fiabilidad, y **la primera que dé un resultado válido gana**:

1. **`lat`/`lon`** del evento (107 eventos): geocodificación inversa con Nominatim a nivel de municipio.
2. **`lugar`** del catálogo (los mismos 107 eventos; sirve de respaldo si la inversa falla): texto libre tipo `LA ALMARCHA, CUENCA` o `Beteta`, geocodificación directa.
3. **Nombre del evento** (los ~272 restantes): mismo patrón de `cronorunner`/`youevent`/`championsxip` -- aislar un candidato a población dentro del nombre (`"6 - MONTIEL - OPEN CR BTT 2023"` -> `Montiel`; `"SAN SILVESTRE MEMBRILLA MANGA 1 - 2021"` -> `Membrilla`). Aquí el nombre trae mucho más ruido que en `cronorunner` (circuitos ciclistas, nombres de club...), y Nominatim siempre devuelve *algo* aunque el candidato sea basura. Por eso, **solo para este caso**, exigimos que el municipio devuelto aparezca dentro de lo que hemos buscado (o al revés); si no, se descarta y la ubicación queda en `NaN` -- no forzamos ningún valor inventado.

Como todos los eventos con coordenadas están en un mismo rincón de Castilla-La Mancha (Cuenca y alrededores), las consultas por nombre llevan un `viewbox` **blando** (`bounded=False`) alrededor de esas coordenadas: Nominatim lo usa solo para desempatar homónimos (p.ej. `Priego`, en Cuenca y en Córdoba), sin descartar resultados de fuera.

In [7]:
_RE_NUMERAL_INICIAL = re.compile(
    r"^\s*(?:[ivxlcdm]+|\d+)\s*[ºª]?\.?(?:er|do|ro|a)?\s*[-–]?\s+",
    re.IGNORECASE,
)
_RE_ANIO = re.compile(r"\b(19|20)\d{2}\b")
_RE_PARENTESIS = re.compile(r"\(.*?\)")
_RE_SAN_SILVESTRE = re.compile(r"^\s*sant?\s+silvestre\b", re.IGNORECASE)
_RE_DISTANCIA_FINAL = re.compile(r"\s+\d+(?:[.,]\d+)?\s?k(?:m)?\.?$", re.IGNORECASE)

_PALABRAS_GENERICAS = {
    "carrera", "correr", "corren", "cross", "trail", "trailkids", "maraton", "marat\u00f3n", "mitja", "media", "medio",
    "milla", "popular", "populars", "urbana", "urbano", "semi", "escolar", "memorial", "trofeo",
    "circuito", "circuit", "campeonato", "cam", "prov", "duatlon", "duatl\u00f3n", "triatlon", "triatl\u00f3n",
    "nocturna", "nocturno", "solidaria", "solidario", "btt", "mtb", "bike", "ruta", "gran", "premio", "fons", "fondo",
    "edicion", "edici\u00f3n", "subida", "pujada", "vuelta", "volta", "copa", "liga", "internacional",
    "provincial", "final", "comarcal", "nacional", "fase", "previa", "pedestre", "peu", "cursa",
    "ayuntamiento", "villa", "ciudad", "ciutat", "fiesta", "fiestas", "aniversario",
    "silvestre", "vertical", "km", "k", "kv", "kilometro", "kil\u00f3metro", "legua", "traves\u00eda",
    "travesia", "travessia", "equipos", "equips", "y", "i", "gala", "open", "cr", "cu", "ciclismo", "marcha",
    "cicloturista", "ciclista", "tour", "cycling", "challenge", "cto", "campeonato", "manga",
    "largo", "larga", "corta", "corto", "nocturnos", "nocturnas", "individual", "ind", "ebike", "e-bike", "relevos",
}
_PREPOSICIONES = {"de", "del", "d\'", "a", "en", "al", "la", "les", "el", "els"}


def candidato_lugar(nombre: str) -> str:
    """Aísla un candidato a población dentro de un fragmento de nombre de evento, quitando
    numerales, año, paréntesis y palabras genéricas de carrera por delante."""
    if not isinstance(nombre, str) or not nombre.strip():
        return ""

    texto = nombre.strip()
    texto = _RE_NUMERAL_INICIAL.sub("", texto)
    texto = _RE_SAN_SILVESTRE.sub("", texto).strip()   # "San" solo es genérico en "San Silvestre" (no en "San Clemente")
    texto = _RE_ANIO.split(texto)[0]
    texto = _RE_PARENTESIS.sub("", texto)
    texto = _RE_DISTANCIA_FINAL.sub("", texto)
    texto = texto.strip(" -–,.")

    tokens = texto.split()
    i = 0
    while i < len(tokens):
        tok = tokens[i].lower().strip(",.-\'\u2019\"")
        if tok in _PALABRAS_GENERICAS or tok in _PREPOSICIONES or re.fullmatch(r"\d+([.,]\d+)?k?m?", tok):
            i += 1
        else:
            break

    candidato = " ".join(tokens[i:]).strip(" ,.-–")
    return candidato if i < len(tokens) else ""


def consultas_desde_nombre(nombre: str, max_consultas: int = 6) -> list[str]:
    """Candidatos a población, de más a menos probable: cada fragmento separado por ' - '
    (el pueblo suele ir en medio: "6 - MONTIEL - OPEN CR BTT"), y para cada uno el texto
    completo y sus recortes por el final y por delante."""
    consultas = []

    def _añadir(q):
        q = q.strip(" ,.-–")
        if len(re.sub(r"[^A-Za-zÀ-ÿ]", "", q)) >= 3 and q not in consultas:
            consultas.append(q)

    fragmentos = [f for f in re.split(r"\s+[-–]\s+", nombre) if f.strip()] if isinstance(nombre, str) else []
    for frag in fragmentos or [nombre]:
        cand = candidato_lugar(frag)
        tokens = cand.split()
        if not tokens:
            continue
        _añadir(cand)
        for n in range(len(tokens) - 1, 0, -1):
            _añadir(" ".join(tokens[:n]))       # sin las últimas palabras ("MEMBRILLA MANGA 1" -> "MEMBRILLA")
        for n in range(1, len(tokens) - 1):
            _añadir(" ".join(tokens[n:]))       # sin las primeras
    return consultas[:max_consultas]


_eventos = df.drop_duplicates("id_timingsys")
_sin_coord_ni_lugar = _eventos[_eventos["lat"].isna() & _eventos["lugar"].isna()]
print(f"{len(_eventos)} eventos: {_eventos['lat'].notna().sum()} con coordenadas, "
      f"{_eventos['lugar'].notna().sum()} con lugar, {len(_sin_coord_ni_lugar)} sin ninguno de los dos")
print("Ejemplos nombre_evento -> consultas de geocodificación:")
for nombre in _sin_coord_ni_lugar["nombre_evento"].sample(12, random_state=0):
    print(f"  {nombre!r} -> {consultas_desde_nombre(nombre)}")

379 eventos: 107 con coordenadas, 107 con lugar, 272 sin ninguno de los dos
Ejemplos nombre_evento -> consultas de geocodificación:
  'V SENDAS DE CUENCA 2022 - MARATÓN' -> ['SENDAS DE CUENCA', 'SENDAS DE', 'SENDAS', 'DE CUENCA']
  'XI Carrera MTB Virgen de Rus - LARGO' -> ['Virgen de Rus', 'Virgen de', 'Virgen', 'de Rus']
  'VII C.P. DE LA GUITARRA ESPAÑOLA - 2023' -> ['C.P. DE LA GUITARRA ESPAÑOLA', 'C.P. DE LA GUITARRA', 'C.P. DE LA', 'C.P. DE', 'DE LA GUITARRA ESPAÑOLA', 'LA GUITARRA ESPAÑOLA']
  '10K NOCTURNA VILLA DE SAN CLEMENTE - 2024' -> ['SAN CLEMENTE', 'SAN']
  'VI CARRERA POPULAR TIERRA DEL VÍTOR - HORCAJO' -> ['TIERRA DEL VÍTOR', 'TIERRA DEL', 'TIERRA', 'DEL VÍTOR', 'HORCAJO']
  'VI MARCHA MTB TIERRA DE VÍTOR -- LARGA' -> ['TIERRA DE VÍTOR -- LARGA', 'TIERRA DE VÍTOR', 'TIERRA DE', 'TIERRA', 'DE VÍTOR -- LARGA', 'VÍTOR -- LARGA']
  'III CARRERA POPULAR TU PUEBLO AMIGO - 2024' -> ['TU PUEBLO AMIGO', 'TU PUEBLO', 'PUEBLO AMIGO']
  'XC SERRANÍA CONQUENSE - LARGA' -> ['SERRANÍ

Guardamos un checkpoint cada 25 eventos en `timingsys_ubicaciones.csv` para poder retomar si se corta (una consulta por segundo como mínimo, por la política de uso de Nominatim: son unos 380 eventos y hasta 6 consultas por evento sin ubicación directa, así que puede tardar un buen rato). La tabla guarda además el `origen` de cada ubicación (`coordenadas`, `lugar`, `nombre` o `sin_resultado`) para poder auditar cuántas salen de cada vía.

In [9]:
from pathlib import Path
import time as _time
import ssl
import unicodedata
import certifi
from geopy.geocoders import Nominatim
from geopy.exc import GeopyError

RUTA_UBICACIONES = Path("../../data/raw/timingsys/timingsys_ubicaciones.csv")

# El Python de python.org en este Mac no trae un almacén de certificados de
# sistema enlazado, así que geopy falla con SSLCertVerificationError sin esto.
_SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())


def _norm(texto) -> str:
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii").upper()
    return re.sub(r"[^A-Z0-9]+", " ", t).strip()


def _extraer_direccion(ubic):
    direccion = ubic.raw.get("address", {})
    municipio = (
        direccion.get("city") or direccion.get("town") or direccion.get("village")
        or direccion.get("municipality")
    )
    comarca = direccion.get("county") or direccion.get("state_district")
    provincia = direccion.get("province") or direccion.get("state")
    return municipio, comarca, provincia


def _municipio_coincide(municipio, query) -> bool:
    """Filtro anti-basura para búsquedas por nombre: el municipio devuelto tiene que
    aparecer dentro de lo buscado (o al revés). Sin esto, Nominatim devuelve
    "algo" para cualquier texto, y un candidato ruidoso acabaría con una población falsa."""
    if not municipio:
        return False
    m, q = _norm(municipio), _norm(query)
    return bool(m) and (f" {m} " in f" {q} " or f" {q} " in f" {m} ")


def _viewbox_desde_coordenadas(df_in, margen=1.5):
    con_coord = df_in.dropna(subset=["lat", "lon"])
    if con_coord.empty:
        return None
    return [
        (con_coord["lat"].min() - margen, con_coord["lon"].min() - margen),
        (con_coord["lat"].max() + margen, con_coord["lon"].max() + margen),
    ]


def geocodificar_ubicaciones_timingsys(df_in, pausa_segundos=1.0):
    geolocalizador = Nominatim(user_agent="tfm_carreras_populares_es", ssl_context=_SSL_CONTEXT)
    viewbox = _viewbox_desde_coordenadas(df_in)

    filas = []
    if RUTA_UBICACIONES.exists():
        filas = pd.read_csv(RUTA_UBICACIONES, dtype={"id_timingsys": str}).to_dict("records")
    hechas = {str(f["id_timingsys"]) for f in filas}

    def _directa(query):
        kwargs = dict(exactly_one=True, country_codes="es", addressdetails=True, timeout=10)
        if viewbox:
            kwargs.update(viewbox=viewbox, bounded=False)
        try:
            ubic = geolocalizador.geocode(f"{query}, España", **kwargs)
        except GeopyError:
            ubic = None
        _time.sleep(pausa_segundos)
        return ubic

    def _inversa(lat, lon):
        try:
            ubic = geolocalizador.reverse((lat, lon), exactly_one=True, zoom=10, addressdetails=True, timeout=10)
        except GeopyError:
            ubic = None
        _time.sleep(pausa_segundos)
        return ubic

    def _resolver_evento(ev):
        """Devuelve (origen, query, ubic) de la primera vía que dé un municipio válido."""
        if pd.notna(ev["lat"]) and pd.notna(ev["lon"]):
            ubic = _inversa(ev["lat"], ev["lon"])
            if ubic and _extraer_direccion(ubic)[0]:
                return "coordenadas", f"{ev['lat']},{ev['lon']}", ubic
        if isinstance(ev["lugar"], str) and ev["lugar"].strip(" ,"):
            query = ev["lugar"].strip(" ,")
            ubic = _directa(query)
            if ubic and _extraer_direccion(ubic)[0]:
                return "lugar", query, ubic
        for query in consultas_desde_nombre(ev["nombre_evento"]):
            ubic = _directa(query)
            if ubic and _municipio_coincide(_extraer_direccion(ubic)[0], query):
                return "nombre", query, ubic
        return "sin_resultado", None, None

    eventos = df_in.drop_duplicates("id_timingsys")[["id_timingsys", "nombre_evento", "lat", "lon", "lugar"]]
    pendientes = eventos[~eventos["id_timingsys"].astype(str).isin(hechas)]
    total = len(pendientes)

    for i, (_, ev) in enumerate(pendientes.iterrows(), 1):
        origen, query, ubic = _resolver_evento(ev)
        fila = {"id_timingsys": ev["id_timingsys"], "origen": origen, "query": query,
                "municipio": None, "comarca": None, "provincia": None, "lat": None, "lon": None}
        if ubic:
            fila["municipio"], fila["comarca"], fila["provincia"] = _extraer_direccion(ubic)
            fila["lat"], fila["lon"] = ubic.latitude, ubic.longitude
        filas.append(fila)
        if i % 25 == 0:
            pd.DataFrame(filas).to_csv(RUTA_UBICACIONES, index=False)
            print(f"  [{i}/{total}] checkpoint guardado")

    df_ubicaciones = pd.DataFrame(filas).drop_duplicates("id_timingsys", keep="last")
    df_ubicaciones.to_csv(RUTA_UBICACIONES, index=False)
    return df_ubicaciones


df_ubicaciones_timingsys = geocodificar_ubicaciones_timingsys(df)
df_ubicaciones_timingsys["id_timingsys"] = df_ubicaciones_timingsys["id_timingsys"].astype(df["id_timingsys"].dtype)

df = df.merge(
    df_ubicaciones_timingsys[["id_timingsys", "municipio", "comarca", "provincia"]],
    on="id_timingsys", how="left", validate="many_to_one",
)

print("Origen de la ubicación (por evento):")
print(df_ubicaciones_timingsys["origen"].value_counts())
print()
print(f"municipio resuelto: {df['municipio'].notna().sum()}/{len(df)} filas")
print(f"comarca resuelta: {df['comarca'].notna().sum()}/{len(df)} filas")
print(f"provincia resuelta: {df['provincia'].notna().sum()}/{len(df)} filas")

Origen de la ubicación (por evento):
origen
nombre           152
sin_resultado    120
coordenadas      107
Name: count, dtype: int64

municipio resuelto: 269/394 filas
comarca resuelta: 176/394 filas
provincia resuelta: 269/394 filas


### Esquema común entre las fuentes

In [11]:
df["fuente"] = "timingsys"

_COLUMNAS_COMUNES = [
    "fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
    "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
]
_COLUMNAS_PROPIAS = [c for c in df.columns if c not in _COLUMNAS_COMUNES]
df = df[_COLUMNAS_COMUNES + _COLUMNAS_PROPIAS]

print(df.shape)
df.head(8)

(394, 23)
      fuente  ...                  lugar
0  timingsys  ...                    NaN
1  timingsys  ...                    NaN
2  timingsys  ...                    NaN
3  timingsys  ...    LA ALMARCHA, CUENCA
4  timingsys  ...  FUENTELESPINO DE HARO
5  timingsys  ...  FUENTELESPINO DE HARO
6  timingsys  ...  FUENTELESPINO DE HARO
7  timingsys  ...                    NaN

[8 rows x 23 columns]


In [12]:
RUTA_LIMPIO = "../../data/processed/timingsys/DF_TIMINGSYS_LIMPIO.csv"
Path(RUTA_LIMPIO).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_LIMPIO, index=False, encoding="utf-8-sig")
print("Guardado:", RUTA_LIMPIO, df.shape)

Guardado: ../../data/processed/timingsys/DF_TIMINGSYS_LIMPIO.csv (394, 23)


### Prueba con filas reales

Antes de fiarnos del resultado con la fuente completa, probamos las funciones que **no** necesitan red (`limpiar_nombre_carrera`, `derivar_distancia_km`, `derivar_fecha_y_dia_semana`, `clasificar_tipo_modalidad`, `clasificar_publico`, `consultas_desde_nombre`) contra filas reales del CSV, con el resultado esperado escrito a mano a partir del nombre y la modalidad.

In [14]:
_df_prueba = pd.DataFrame({
    "nombre_carrera": [
        "XXI CP SANTA EULALIA - VILLARES DEL SAZ - 2026 ",
        "XIV MARCHA MTB V. FUENTELESPINO DE HARO 2026 - EBIKE",
        "CAM. PROV. CICLISMO EN RUTA  - ARGUISUELAS - ALEVÍNES",
        "IX TRIATLON CROSS VILLA DE LAS MESAS 2023 - INDIVIDUAL",
        "TRAIL LA OSERA 48K",
        "SAN SILVESTRE MESEÑA 23 - 5,3 KM",
    ],
    "modalidad": ["A PIE", "BICI (MTB/BTT)", "BICI (MTB/BTT)", "DUATLON/TRIATLON", "TRAIL", "A PIE"],
    "distancia_metros": [9000, 40000, None, 27000, 48000, 5300],
    "fecha_evento": ["2026-09-05T00:00:00", "2026-08-09T00:00:00", "2023-05-07T00:00:00",
                     "2023-06-03T00:00:00", "2023-04-01T00:00:00", "2023-12-31T00:00:00"],
})

_df_prueba["nombre_carrera"] = limpiar_nombre_carrera(_df_prueba["nombre_carrera"])
_df_prueba["distancia"] = derivar_distancia_km(_df_prueba)
_df_prueba["fecha"], _df_prueba["dia_semana"] = derivar_fecha_y_dia_semana(_df_prueba)
_df_prueba["tipo_modalidad"] = [
    clasificar_tipo_modalidad(n, m) for n, m in zip(_df_prueba["nombre_carrera"], _df_prueba["modalidad"])
]
_df_prueba["publico"] = _df_prueba["nombre_carrera"].apply(clasificar_publico)

assert _df_prueba["nombre_carrera"].iloc[0] == "XXI CP SANTA EULALIA - VILLARES DEL SAZ - 2026"
assert _df_prueba["nombre_carrera"].iloc[2] == "CAM. PROV. CICLISMO EN RUTA - ARGUISUELAS - ALEVÍNES"
assert list(_df_prueba["distancia"]) == [9.0, 40.0, 0.0, 27.0, 48.0, 5.3]
assert list(_df_prueba["tipo_modalidad"]) == [
    "road running", "Ciclismo y btt", "Ciclismo y btt", "Multidisciplina", "trail running", "road running",
]
assert list(_df_prueba["publico"]) == [
    "Absoluta/General", "Absoluta/General", "Infantil", "Absoluta/General", "Absoluta/General", "Absoluta/General",
]
assert list(_df_prueba["dia_semana"]) == ["Sábado", "Domingo", "Domingo", "Sábado", "Sábado", "Domingo"]

assert "SAN CLEMENTE" in consultas_desde_nombre("10K NOCTURNA VILLA DE SAN CLEMENTE - 2024")
assert "UÑA" in consultas_desde_nombre("XVIII CARRERA POPULAR DE UÑA - 2023")
assert consultas_desde_nombre("SAN SILVESTRE MEMBRILLA MANGA 1 - 2021")[0] == "MEMBRILLA MANGA 1"
assert "MONTIEL" in consultas_desde_nombre("6 - MONTIEL - OPEN CR BTT 2023")
assert "MEMBRILLA" in consultas_desde_nombre("SAN SILVESTRE MEMBRILLA MANGA 1 - 2021")
assert "VILLARES DEL SAZ" in consultas_desde_nombre("XXI CP SANTA EULALIA - VILLARES DEL SAZ - 2026")
assert _municipio_coincide("Membrilla", "MEMBRILLA MANGA 1") and not _municipio_coincide("Madrid", "MEMBRILLA MANGA 1")

print("Todo OK: las funciones de limpieza se comportan bien contra las filas de prueba.")
_df_prueba[["nombre_carrera", "distancia", "dia_semana", "tipo_modalidad", "publico"]]

Todo OK: las funciones de limpieza se comportan bien contra las filas de prueba.
                                      nombre_carrera  ...           publico
0     XXI CP SANTA EULALIA - VILLARES DEL SAZ - 2026  ...  Absoluta/General
1  XIV MARCHA MTB V. FUENTELESPINO DE HARO 2026 -...  ...  Absoluta/General
2  CAM. PROV. CICLISMO EN RUTA - ARGUISUELAS - AL...  ...          Infantil
3  IX TRIATLON CROSS VILLA DE LAS MESAS 2023 - IN...  ...  Absoluta/General
4                                 TRAIL LA OSERA 48K  ...  Absoluta/General
5                   SAN SILVESTRE MESEÑA 23 - 5,3 KM  ...  Absoluta/General

[6 rows x 5 columns]
